# 7. The t-Test

**Statistical Foundations for Data Science — Notebook 7 of 8**

The t-test is the most-used hypothesis test in the world. It answers one question in three
different situations: **is this mean different from what I expected?**

It exists because of a practical problem. The z-test needs the population standard deviation
$\sigma$, and you never know it. Replacing $\sigma$ with the sample estimate $s$ adds extra
uncertainty — and in 1908 William Sealy Gosset, a brewer at Guinness publishing under the
pen name "Student", worked out exactly how much. Hence *Student's t*.

### What you will learn

1. Why $t$ and not $z$, and what degrees of freedom mean
2. **One-sample t-test** — a mean against a target value
3. **Two-sample (independent) t-test** — comparing two groups
4. **Welch's t-test** — the version you should use by default
5. **Paired t-test** — before/after on the same units
6. Checking the **assumptions**, and what to do when they fail
7. **Confidence intervals** and **effect sizes** for each test
8. A full A/B-test walkthrough from data to recommendation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

rng = np.random.default_rng(seed=314)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)

---
## 7.1 Why t instead of z?

With $\sigma$ known:

$$z = \frac{\bar{x} - \mu_0}{\sigma/\sqrt{n}} \sim N(0, 1)$$

With $\sigma$ **unknown**, replaced by the sample sd $s$:

$$t = \frac{\bar{x} - \mu_0}{s/\sqrt{n}} \sim t_{n-1}$$

Because $s$ is itself random, $t$ varies more than $z$ — hence the **heavier tails**. The
practical consequence: critical values are larger, so you need stronger evidence to reject
$H_0$. As $n$ grows, $s \to \sigma$ and the t-distribution converges to the Normal.

### Degrees of freedom

$df = n - 1$ for one sample. Intuition: once you know $\bar{x}$ and $n-1$ of the
observations, the last one is determined — you "spent" one degree of freedom estimating the
mean.

In [ ]:
print(f"{'df':>6} {'t critical (95%)':>18} {'z critical':>12} {'excess':>9}")
z_crit = stats.norm.ppf(0.975)
for df in (1, 2, 5, 10, 20, 30, 60, 120, 1000):
    t_crit = stats.t(df).ppf(0.975)
    print(f"{df:>6} {t_crit:>18.4f} {z_crit:>12.4f} {(t_crit/z_crit - 1)*100:>8.1f}%")
print("\nBy df = 30 the penalty is under 5%; by df = 120 it is negligible.")
print("This is where the folklore 'n > 30 means you can use z' comes from.")

In [ ]:
# Why the tails matter: using z when you should use t inflates your false-positive rate.
alpha, reps = 0.05, 20_000
print("Empirical Type I error rate when H0 is true:")
print(f"{'n':>5} {'using t (correct)':>20} {'using z (wrong)':>18}")
for n in (5, 10, 20, 50):
    samples = rng.normal(0, 1, size=(reps, n))
    xbar = samples.mean(axis=1)
    s = samples.std(axis=1, ddof=1)
    tstat = xbar / (s / np.sqrt(n))
    rate_t = (np.abs(tstat) > stats.t(n - 1).ppf(1 - alpha/2)).mean()
    rate_z = (np.abs(tstat) > stats.norm.ppf(1 - alpha/2)).mean()
    print(f"{n:>5} {rate_t:>20.4f} {rate_z:>18.4f}")
print(f"\nTarget is {alpha}. The z column is too high for small n -- you would reject")
print("true null hypotheses more often than you promised.")

---
## 7.2 One-sample t-test

**Question:** is the population mean different from a specified value $\mu_0$?

$$H_0: \mu = \mu_0 \qquad H_1: \mu \ne \mu_0$$

$$t = \frac{\bar{x} - \mu_0}{s/\sqrt{n}}, \qquad df = n - 1$$

**Assumptions:** (1) observations independent, (2) the data (or the sample mean) is
approximately Normal, (3) the variable is continuous.

**Confidence interval for the mean:**

$$\bar{x} \pm t_{\alpha/2,\,n-1}\,\frac{s}{\sqrt{n}}$$

**Example.** A cereal manufacturer claims boxes contain 500 g. A consumer group weighs 25
boxes.

In [ ]:
boxes = np.array([496.2, 501.4, 494.8, 499.1, 492.5, 497.8, 503.2, 495.6, 498.3, 491.9,
                  500.5, 496.7, 493.4, 499.8, 494.1, 497.2, 502.6, 495.0, 498.9, 493.7,
                  496.5, 500.1, 494.4, 497.6, 492.8])

mu_0 = 500.0
n = len(boxes)
xbar, s = boxes.mean(), boxes.std(ddof=1)
se = s / np.sqrt(n)
t_stat = (xbar - mu_0) / se
df = n - 1
p_val = 2 * stats.t(df).sf(abs(t_stat))

print(f"n = {n},  sample mean = {xbar:.3f} g,  sample sd = {s:.3f} g")
print(f"standard error = {se:.4f}")
print(f"t = ({xbar:.3f} - {mu_0}) / {se:.4f} = {t_stat:.4f}   with df = {df}")
print(f"two-sided p-value = {p_val:.6f}")
print()
print("scipy, in one line:")
res = stats.ttest_1samp(boxes, popmean=mu_0)
print(f"  t = {res.statistic:.4f}, p = {res.pvalue:.6f}, df = {res.df}")

In [ ]:
# Confidence interval and effect size
t_crit = stats.t(df).ppf(0.975)
ci = (xbar - t_crit*se, xbar + t_crit*se)
d = (xbar - mu_0) / s

print(f"95% CI for the true mean weight : [{ci[0]:.3f}, {ci[1]:.3f}] g")
print(f"Does the CI contain 500 g?      : {'yes' if ci[0] <= mu_0 <= ci[1] else 'NO'}")
print(f"Cohen's d = ({xbar:.2f} - {mu_0})/{s:.2f} = {d:.3f}  (a large effect)")
print()
print("scipy can produce the interval directly:")
print(f"  {res.confidence_interval(0.95)}")
print()
print("CONCLUSION: boxes are underfilled by about "
      f"{mu_0 - xbar:.1f} g on average (95% CI {mu_0-ci[1]:.1f} to {mu_0-ci[0]:.1f} g short).")
print("With p < 0.001 this is not chance variation. It is also practically meaningful:")
print(f"a {(mu_0-xbar)/mu_0*100:.1f}% shortfall across a production run is a real cost to consumers.")

In [ ]:
# Visualise the test
fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.2))

ax[0].hist(boxes, bins=10, color="steelblue", edgecolor="white")
ax[0].axvline(mu_0, color="crimson", lw=2, label=f"claimed {mu_0} g")
ax[0].axvline(xbar, color="darkgreen", lw=2, label=f"observed mean {xbar:.1f} g")
ax[0].axvspan(ci[0], ci[1], color="darkgreen", alpha=0.15, label="95% CI")
ax[0].set_xlabel("weight (g)"); ax[0].set_title("The data"); ax[0].legend(fontsize=8)

xs = np.linspace(-5, 5, 600)
ax[1].plot(xs, stats.t(df).pdf(xs), color="black")
tail = xs <= -abs(t_stat)
ax[1].fill_between(xs[tail], stats.t(df).pdf(xs[tail]), color="crimson", alpha=0.6)
tail2 = xs >= abs(t_stat)
ax[1].fill_between(xs[tail2], stats.t(df).pdf(xs[tail2]), color="crimson", alpha=0.6)
ax[1].axvline(t_stat, color="crimson", lw=2, ls="--", label=f"t = {t_stat:.2f}")
ax[1].set_title(f"Null distribution t({df}), p = {p_val:.2e}")
ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

---
## 7.3 Two-sample (independent) t-test

**Question:** do two independent groups have different means?

$$H_0: \mu_1 = \mu_2 \qquad H_1: \mu_1 \ne \mu_2$$

There are two versions, and the choice matters.

### Student's t (pooled variance) — assumes $\sigma_1 = \sigma_2$

$$t = \frac{\bar{x}_1 - \bar{x}_2}{s_p\sqrt{\frac{1}{n_1}+\frac{1}{n_2}}}, \qquad
s_p^2 = \frac{(n_1-1)s_1^2 + (n_2-1)s_2^2}{n_1+n_2-2}, \qquad df = n_1+n_2-2$$

### Welch's t — does **not** assume equal variances

$$t = \frac{\bar{x}_1 - \bar{x}_2}{\sqrt{\frac{s_1^2}{n_1}+\frac{s_2^2}{n_2}}}, \qquad
df = \frac{\left(\frac{s_1^2}{n_1}+\frac{s_2^2}{n_2}\right)^2}
{\frac{(s_1^2/n_1)^2}{n_1-1}+\frac{(s_2^2/n_2)^2}{n_2-1}}$$

> **Recommendation: default to Welch's test.** It costs almost nothing in power when the
> variances *are* equal, and it protects you when they are not. Note that
> `scipy.stats.ttest_ind` defaults to `equal_var=True`, so you must pass
> `equal_var=False` explicitly.

In [ ]:
# Two teaching methods, unequal group sizes and unequal spread
traditional = rng.normal(72, 8, 45)
interactive = rng.normal(78, 13, 32)

print(f"Traditional : n={len(traditional)}, mean={traditional.mean():.2f}, sd={traditional.std(ddof=1):.2f}")
print(f"Interactive : n={len(interactive)}, mean={interactive.mean():.2f}, sd={interactive.std(ddof=1):.2f}")
print(f"Variance ratio: {interactive.var(ddof=1)/traditional.var(ddof=1):.2f}\n")

student = stats.ttest_ind(interactive, traditional, equal_var=True)
welch   = stats.ttest_ind(interactive, traditional, equal_var=False)

print(f"Student's t (pooled) : t = {student.statistic:.4f}, df = {student.df:.1f}, p = {student.pvalue:.5f}")
print(f"Welch's t            : t = {welch.statistic:.4f}, df = {welch.df:.1f}, p = {welch.pvalue:.5f}")
print("\nWelch spends some degrees of freedom to buy robustness.")

In [ ]:
# Compute Welch by hand so the formula is not a black box
n1, n2 = len(interactive), len(traditional)
m1, m2 = interactive.mean(), traditional.mean()
v1, v2 = interactive.var(ddof=1), traditional.var(ddof=1)

se_w = np.sqrt(v1/n1 + v2/n2)
t_w = (m1 - m2) / se_w
df_w = (v1/n1 + v2/n2)**2 / ((v1/n1)**2/(n1-1) + (v2/n2)**2/(n2-1))

print(f"SE     = sqrt({v1:.2f}/{n1} + {v2:.2f}/{n2}) = {se_w:.4f}")
print(f"t      = ({m1:.2f} - {m2:.2f}) / {se_w:.4f} = {t_w:.4f}")
print(f"df     = {df_w:.3f}  (Welch-Satterthwaite)")
print(f"p      = {2*stats.t(df_w).sf(abs(t_w)):.6f}")

# CI for the difference in means
tc = stats.t(df_w).ppf(0.975)
diff = m1 - m2
print(f"\nDifference = {diff:+.2f} marks, 95% CI [{diff - tc*se_w:+.2f}, {diff + tc*se_w:+.2f}]")

def cohens_d(a, b):
    na, nb = len(a), len(b)
    sp = np.sqrt(((na-1)*a.var(ddof=1) + (nb-1)*b.var(ddof=1)) / (na+nb-2))
    return (a.mean() - b.mean()) / sp

print(f"Cohen's d  = {cohens_d(interactive, traditional):.3f}")

In [ ]:
# Why equal_var=True is risky: unequal variances plus unequal group sizes inflates alpha
reps = 4_000
print("Type I error rate when H0 is TRUE but variances differ 4-fold:")
print(f"{'n1':>5} {'n2':>5} {'Student p<0.05':>16} {'Welch p<0.05':>15}")
for n1_, n2_ in [(10, 10), (10, 40), (40, 10), (30, 30)]:
    s_rej = w_rej = 0
    for _ in range(reps):
        a = rng.normal(0, 1, n1_)
        b = rng.normal(0, 4, n2_)              # same mean, 4x the sd
        s_rej += stats.ttest_ind(a, b, equal_var=True).pvalue < 0.05
        w_rej += stats.ttest_ind(a, b, equal_var=False).pvalue < 0.05
    print(f"{n1_:>5} {n2_:>5} {s_rej/reps:>16.4f} {w_rej/reps:>15.4f}")
print("\nStudent's test drifts away from 0.05 when group sizes and variances are both")
print("unbalanced. Welch stays close. This is why Welch should be your default.")

In [ ]:
# Visual comparison of the two groups
fig, ax = plt.subplots(1, 3, figsize=(15, 4))

ax[0].hist(traditional, bins=14, alpha=0.65, label="traditional", color="steelblue")
ax[0].hist(interactive, bins=14, alpha=0.65, label="interactive", color="seagreen")
ax[0].set_title("Distributions"); ax[0].legend(fontsize=8); ax[0].set_xlabel("marks")

frame = pd.DataFrame({
    "marks": np.r_[traditional, interactive],
    "method": ["traditional"]*len(traditional) + ["interactive"]*len(interactive),
})
sns.boxplot(data=frame, x="method", y="marks", ax=ax[1], hue="method", legend=False)
ax[1].set_title("Box plot")

sns.violinplot(data=frame, x="method", y="marks", ax=ax[2], hue="method", legend=False, inner="quartile")
ax[2].set_title("Violin plot (shape + quartiles)")
plt.tight_layout(); plt.show()

---
## 7.4 Paired t-test

**Question:** for the *same* units measured twice, is the average change non-zero?

$$H_0: \mu_d = 0 \qquad H_1: \mu_d \ne 0 \qquad
t = \frac{\bar{d}}{s_d/\sqrt{n}}, \qquad df = n - 1$$

where $d_i = x_{i,\text{after}} - x_{i,\text{before}}$. **It is literally a one-sample
t-test on the differences.**

**Use it when** each observation in group 1 has a natural partner in group 2: before/after
on the same person, left/right eye, twins, matched customers, the same server under two
configurations.

**Why it matters:** pairing removes between-subject variability, which is often the largest
source of noise. The gain in power can be dramatic — as the demonstration below shows.

In [ ]:
# 20 employees take a productivity test before and after training.
# Individual baseline ability varies a lot -- that is the noise pairing removes.
baseline = rng.normal(60, 15, 20)                       # big person-to-person differences
before = baseline + rng.normal(0, 3, 20)
after  = baseline + 5 + rng.normal(0, 3, 20)            # true training effect = +5

paired = stats.ttest_rel(after, before)
unpaired = stats.ttest_ind(after, before, equal_var=False)

d = after - before
print(f"Mean before : {before.mean():.2f}   Mean after : {after.mean():.2f}")
print(f"Mean change : {d.mean():+.2f}  (sd of changes {d.std(ddof=1):.2f})")
print(f"sd of raw scores: {before.std(ddof=1):.2f}   <- much larger than the sd of changes\n")
print(f"PAIRED   t-test : t = {paired.statistic:.4f}, df = {paired.df}, p = {paired.pvalue:.6f}")
print(f"UNPAIRED t-test : t = {unpaired.statistic:.4f}, df = {unpaired.df:.1f}, p = {unpaired.pvalue:.6f}")
print("\nSame data, same true effect. Ignoring the pairing throws away the information")
print("that lets you see it -- the unpaired test is drowned in between-person variation.")

# Equivalence with the one-sample test on differences
print(f"\nOne-sample t-test on the differences: "
      f"t = {stats.ttest_1samp(d, 0).statistic:.4f}, p = {stats.ttest_1samp(d, 0).pvalue:.6f}")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))

for i in range(20):
    ax[0].plot([0, 1], [before[i], after[i]], "o-", color="grey", alpha=0.6, ms=4)
ax[0].plot([0, 1], [before.mean(), after.mean()], "o-", color="crimson", lw=3, ms=9, label="means")
ax[0].set_xticks([0, 1]); ax[0].set_xticklabels(["before", "after"])
ax[0].set_ylabel("productivity score"); ax[0].set_title("Every line is one employee")
ax[0].legend(fontsize=8)

ax[1].hist(d, bins=10, color="steelblue", edgecolor="white")
ax[1].axvline(0, color="black", lw=1.5, label="no change")
ax[1].axvline(d.mean(), color="crimson", lw=2, label=f"mean change {d.mean():+.2f}")
ax[1].set_xlabel("after - before"); ax[1].set_title("The differences are what we test")
ax[1].legend(fontsize=8)

ax[2].scatter(before, after, s=40, color="steelblue")
lims = [min(before.min(), after.min()) - 2, max(before.max(), after.max()) + 2]
ax[2].plot(lims, lims, "k--", label="no change line")
ax[2].set_xlabel("before"); ax[2].set_ylabel("after")
ax[2].set_title(f"Correlated pairs (r = {np.corrcoef(before, after)[0,1]:.3f})")
ax[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

print("The high before/after correlation is exactly why pairing pays off.")
print("The stronger the pairing, the bigger the power gain.")

In [ ]:
# Quantify the power gain from pairing across correlation levels
def power_gain(corr, n=20, effect=5, sd=15, reps=3_000):
    p_paired = p_unpaired = 0
    for _ in range(reps):
        base = rng.normal(60, sd, n)
        noise_sd = sd * np.sqrt(max(1e-9, 1/corr - 1)) if corr < 1 else 0
        b = base + rng.normal(0, noise_sd, n)
        a = base + effect + rng.normal(0, noise_sd, n)
        p_paired   += stats.ttest_rel(a, b).pvalue < 0.05
        p_unpaired += stats.ttest_ind(a, b, equal_var=False).pvalue < 0.05
    return p_paired/reps, p_unpaired/reps

print(f"{'pairing corr':>14} {'paired power':>14} {'unpaired power':>16}")
for c in (0.3, 0.6, 0.9, 0.98):
    pp, pu = power_gain(c)
    print(f"{c:>14.2f} {pp:>14.3f} {pu:>16.3f}")
print("\nWhen the pairs are barely related, pairing gains little. When they are tightly")
print("related (repeated measures on the same unit), pairing is transformative.")

---
## 7.5 Checking the assumptions

Every t-test rests on three assumptions. Check them, in this order of importance:

**1. Independence** — the most important and the least testable. It is a property of your
*data collection*, not your data. Repeated measures on the same subject treated as
independent, or clustered data ignored, will invalidate the test no matter how Normal it
looks.

**2. Normality** — of the *sampling distribution of the mean*, not the raw data. Thanks to
the CLT this is forgiving: with $n > 30$ per group and no extreme skew you are usually fine.
Check with a Q–Q plot; use Shapiro–Wilk as a supplement, not an oracle.

**3. Equal variances** — only needed for Student's pooled test. Use Welch and stop worrying.
Levene's test can check it if you must.

In [ ]:
def assumption_report(*groups, names=None):
    '''Run the standard assumption checks for a t-test.'''
    names = names or [f"group {i+1}" for i in range(len(groups))]
    print("NORMALITY (Shapiro-Wilk; p > 0.05 is consistent with Normal)")
    for g, nm in zip(groups, names):
        w, p = stats.shapiro(g)
        print(f"  {nm:<14} n={len(g):<4} W={w:.4f}  p={p:.4f}  "
              f"skew={stats.skew(g):+.2f}  {'OK' if p > 0.05 else 'QUESTIONABLE'}")
    if len(groups) == 2:
        lev = stats.levene(*groups)
        bar = stats.bartlett(*groups)
        print("\nEQUAL VARIANCES")
        print(f"  Levene   W={lev.statistic:.4f}  p={lev.pvalue:.4f}  "
              f"{'equal' if lev.pvalue > 0.05 else 'UNEQUAL -> use Welch'}")
        print(f"  Bartlett W={bar.statistic:.4f}  p={bar.pvalue:.4f}  (assumes normality itself)")
        print(f"  variance ratio = {max(g.var(ddof=1) for g in groups)/min(g.var(ddof=1) for g in groups):.2f}")

assumption_report(traditional, interactive, names=["traditional", "interactive"])

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
stats.probplot(traditional, dist="norm", plot=ax[0])
ax[0].set_title("Q-Q plot: traditional group")
stats.probplot(interactive, dist="norm", plot=ax[1])
ax[1].set_title("Q-Q plot: interactive group")
plt.tight_layout(); plt.show()
print("Points near the line -> the Normality assumption is reasonable.")

### When the assumptions fail

| Problem | Fix |
|---|---|
| Skewed data | Log or Box–Cox transform, then t-test on the transformed scale |
| Outliers | Investigate them; consider a trimmed mean or a rank test |
| Non-Normal, small $n$ | **Mann–Whitney U** (independent) or **Wilcoxon signed-rank** (paired) |
| Unequal variances | **Welch's t-test** |
| Any of the above | **Permutation test** (Notebook 6) — assumption-light and exact |
| Dependent observations | Mixed-effects model / clustered standard errors |

Note that Mann–Whitney tests a *shift in distribution*, not strictly the mean, so its
conclusion is subtly different. Report which test you used and why.

In [ ]:
# Compare the options on badly skewed data
skew_a = rng.lognormal(3.2, 0.85, 35)
skew_b = rng.lognormal(3.5, 0.85, 35)

print(f"Group A: mean {skew_a.mean():.1f}, median {np.median(skew_a):.1f}, skew {stats.skew(skew_a):+.2f}")
print(f"Group B: mean {skew_b.mean():.1f}, median {np.median(skew_b):.1f}, skew {stats.skew(skew_b):+.2f}")
print(f"Shapiro on A: p = {stats.shapiro(skew_a).pvalue:.5f}  -> not Normal\n")

options = {
    "Welch t on raw data":     stats.ttest_ind(skew_a, skew_b, equal_var=False).pvalue,
    "Welch t on log data":     stats.ttest_ind(np.log(skew_a), np.log(skew_b), equal_var=False).pvalue,
    "Mann-Whitney U":          stats.mannwhitneyu(skew_a, skew_b).pvalue,
    "Permutation (means)":     stats.permutation_test(
                                   (skew_a, skew_b), lambda a, b: a.mean() - b.mean(),
                                   permutation_type="independent", n_resamples=10_000,
                                   random_state=0).pvalue,
}
for name, p in options.items():
    print(f"  {name:<24} p = {p:.4f}")
print("\nOn log scale and by rank the signal is clearer, because the raw-scale mean is")
print("dominated by a handful of huge values.")

---
## 7.6 A complete A/B test walkthrough

Let's do the whole job properly: a checkout redesign, measured by revenue per session.
We will follow the five steps from Notebook 6 and produce a recommendation a product
manager could act on.

In [ ]:
# STEP 0: the data (revenue per session, in currency units)
n_ctrl, n_var = 480, 495
control = np.maximum(0, rng.lognormal(3.4, 0.85, n_ctrl))
variant = np.maximum(0, rng.lognormal(3.52, 0.85, n_var))

summary = pd.DataFrame({
    "group": ["control", "variant"],
    "n": [n_ctrl, n_var],
    "mean": [control.mean(), variant.mean()],
    "median": [np.median(control), np.median(variant)],
    "sd": [control.std(ddof=1), variant.std(ddof=1)],
}).round(2)
print(summary.to_string(index=False))
print(f"\nObserved lift in mean revenue: {(variant.mean()/control.mean() - 1)*100:+.2f}%")

In [ ]:
# STEP 1: hypotheses, decided in advance
print("H0: mean revenue per session is the same in both groups")
print("H1: the means differ (two-sided -- a redesign could plausibly hurt)")
print("alpha = 0.05, chosen before looking at the data")
print("Pre-registered decision rule: ship only if the 95% CI lower bound exceeds +2%\n")

# STEP 2: assumptions
assumption_report(control, variant, names=["control", "variant"])
print("\nRevenue is right-skewed, as revenue always is. But n is ~500 per group,")
print("so the CLT covers the sampling distribution of the mean. We proceed with Welch,")
print("and we will cross-check with a permutation test.")

In [ ]:
# STEP 3: the test
res = stats.ttest_ind(variant, control, equal_var=False)
diff = variant.mean() - control.mean()
se = np.sqrt(variant.var(ddof=1)/n_var + control.var(ddof=1)/n_ctrl)
tc = stats.t(res.df).ppf(0.975)
ci_lo, ci_hi = diff - tc*se, diff + tc*se

print(f"Welch t({res.df:.1f}) = {res.statistic:.4f},  p = {res.pvalue:.5f}")
print(f"Difference in means  = {diff:+.3f}")
print(f"95% CI               = [{ci_lo:+.3f}, {ci_hi:+.3f}]")
print(f"Cohen's d            = {cohens_d(variant, control):+.4f}")

# Cross-checks
perm = stats.permutation_test((variant, control), lambda a, b: a.mean() - b.mean(),
                              permutation_type="independent", n_resamples=20_000,
                              random_state=7)
print(f"\nPermutation test p   = {perm.pvalue:.5f}   (agrees with Welch)")
print(f"Mann-Whitney p       = {stats.mannwhitneyu(variant, control).pvalue:.5f}")

In [ ]:
# STEP 4-5: decide and report in business terms
rel_lift = diff / control.mean()
rel_lo, rel_hi = ci_lo / control.mean(), ci_hi / control.mean()

print("=" * 68)
print("A/B TEST REPORT: checkout redesign")
print("=" * 68)
print(f"Control : {n_ctrl} sessions, mean revenue {control.mean():.2f}")
print(f"Variant : {n_var} sessions, mean revenue {variant.mean():.2f}")
print(f"Lift    : {rel_lift:+.2%}  (95% CI {rel_lo:+.2%} to {rel_hi:+.2%})")
print(f"Test    : Welch t({res.df:.0f}) = {res.statistic:.2f}, p = {res.pvalue:.4f}")
print(f"Effect  : Cohen's d = {cohens_d(variant, control):.3f} (small)")
print("-" * 68)
decision = "SHIP" if rel_lo > 0.02 else ("HOLD - inconclusive" if res.pvalue >= 0.05 else "SHIP with monitoring")
print(f"Pre-registered rule (CI lower bound > +2%): lower bound is {rel_lo:+.2%}")
print(f"DECISION: {decision}")
print("-" * 68)
print("Caveats to state alongside the number:")
print("  * revenue is heavy-tailed; the mean is sensitive to a few large orders")
print("  * this covers one time window; seasonality is not accounted for")
print("  * one metric only -- check retention and support load before a full rollout")
print("=" * 68)

In [ ]:
# A bootstrap CI on the RATIO is often what stakeholders actually want
B = 10_000
ratios = np.empty(B)
for i in range(B):
    c = control[rng.integers(0, n_ctrl, n_ctrl)]
    v = variant[rng.integers(0, n_var, n_var)]
    ratios[i] = v.mean() / c.mean() - 1

lo, hi = np.quantile(ratios, [0.025, 0.975])
plt.hist(ratios * 100, bins=70, color="steelblue", edgecolor="none")
plt.axvline(0, color="black", lw=1.5, label="no effect")
plt.axvline(lo*100, color="crimson", ls="--", lw=2)
plt.axvline(hi*100, color="crimson", ls="--", lw=2, label="bootstrap 95% CI")
plt.xlabel("relative lift (%)"); plt.ylabel("frequency")
plt.title("Bootstrap distribution of the relative lift")
plt.legend(); plt.show()

print(f"Bootstrap 95% CI for the lift: {lo:+.2%} to {hi:+.2%}")
print(f"P(lift > 0) across resamples : {(ratios > 0).mean():.3f}")
print(f"P(lift > 2%) across resamples: {(ratios > 0.02).mean():.3f}")

---
## Exercises

**Exercise 1.** A machine is supposed to cut rods to 25.0 cm. A sample of 18 rods gives the
measurements below.
(a) Test at $\alpha = 0.05$ whether the machine is off-target.
(b) Report the 95% CI and Cohen's d.
(c) Check the Normality assumption.
(d) Would your conclusion change at $\alpha = 0.01$?

In [ ]:
# --- Solution 1 -------------------------------------------------------------
rods = np.array([25.12, 24.95, 25.08, 25.21, 24.88, 25.15, 25.03, 25.18, 24.99,
                 25.24, 25.06, 25.11, 24.92, 25.17, 25.09, 25.22, 25.01, 25.14])

target = 25.0
r = stats.ttest_1samp(rods, target)
ci = r.confidence_interval(0.95)
dd = (rods.mean() - target) / rods.std(ddof=1)

print(f"(a) n={len(rods)}, mean={rods.mean():.4f}, sd={rods.std(ddof=1):.4f}")
print(f"    t({r.df}) = {r.statistic:.4f}, p = {r.pvalue:.5f}")
print(f"    Decision at 0.05: {'REJECT H0 -- machine is off-target' if r.pvalue < 0.05 else 'fail to reject'}")
print(f"(b) 95% CI for mean = [{ci.low:.4f}, {ci.high:.4f}]   Cohen's d = {dd:.3f}")
print(f"(c) Shapiro-Wilk p = {stats.shapiro(rods).pvalue:.4f} "
      f"({'Normality OK' if stats.shapiro(rods).pvalue > 0.05 else 'questionable'})")
print(f"(d) At alpha=0.01: {'still reject' if r.pvalue < 0.01 else 'fail to reject'} "
      f"(p = {r.pvalue:.5f})")
print(f"\nPractical note: the bias is only {rods.mean()-target:+.3f} cm. Whether that")
print("matters depends on the engineering tolerance, not on the p-value.")

**Exercise 2.** Two delivery routes are timed. Route A: 20 deliveries. Route B: 20
deliveries with much more variable traffic. Choose and justify a test, run it, and give a
recommendation.

In [ ]:
# --- Solution 2 -------------------------------------------------------------
route_a = rng.normal(34, 4, 20)
route_b = rng.normal(31, 11, 20)

print(f"Route A: mean {route_a.mean():.2f} min, sd {route_a.std(ddof=1):.2f}")
print(f"Route B: mean {route_b.mean():.2f} min, sd {route_b.std(ddof=1):.2f}")
print(f"Variance ratio: {route_b.var(ddof=1)/route_a.var(ddof=1):.2f}\n")

lev = stats.levene(route_a, route_b)
print(f"Levene p = {lev.pvalue:.4f} -> variances "
      f"{'differ' if lev.pvalue < 0.05 else 'are comparable'}; Welch is the right choice.\n")

w = stats.ttest_ind(route_a, route_b, equal_var=False)
d_ = route_a.mean() - route_b.mean()
se_ = np.sqrt(route_a.var(ddof=1)/20 + route_b.var(ddof=1)/20)
tc_ = stats.t(w.df).ppf(0.975)
print(f"Welch t({w.df:.1f}) = {w.statistic:.4f}, p = {w.pvalue:.4f}")
print(f"Difference (A - B) = {d_:+.2f} min, 95% CI [{d_-tc_*se_:+.2f}, {d_+tc_*se_:+.2f}]")
print()
print("RECOMMENDATION: route B is faster on average but far less predictable.")
print(f"  A: 90% of deliveries within {np.percentile(route_a, 90):.0f} min")
print(f"  B: 90% of deliveries within {np.percentile(route_b, 90):.0f} min")
print("For a service with a delivery-time guarantee, the LOW-VARIANCE route may be")
print("preferable even though its mean is worse. A t-test on means alone cannot tell")
print("you that -- you have to look at the spread.")

**Exercise 3.** A clinic measures blood pressure on 15 patients before and after a
medication. Run the appropriate test, and demonstrate what would go wrong if you treated
the two sets of readings as independent samples.

In [ ]:
# --- Solution 3 -------------------------------------------------------------
patient_baseline = rng.normal(148, 18, 15)
bp_before = patient_baseline + rng.normal(0, 4, 15)
bp_after  = patient_baseline - 9 + rng.normal(0, 4, 15)      # true effect: -9 mmHg

changes = bp_after - bp_before
pr = stats.ttest_rel(bp_after, bp_before)
ur = stats.ttest_ind(bp_after, bp_before, equal_var=False)

print("PAIRED is correct: each patient is their own control.")
print(f"  mean change = {changes.mean():+.2f} mmHg (sd {changes.std(ddof=1):.2f})")
print(f"  t({pr.df}) = {pr.statistic:.4f}, p = {pr.pvalue:.6f}")
ci = stats.ttest_1samp(changes, 0).confidence_interval(0.95)
print(f"  95% CI for the change: [{ci.low:+.2f}, {ci.high:+.2f}] mmHg")
print(f"  Cohen's d (paired) = {changes.mean()/changes.std(ddof=1):+.3f}")

print(f"\nUNPAIRED (wrong): t({ur.df:.1f}) = {ur.statistic:.4f}, p = {ur.pvalue:.4f}")
print(f"  Between-patient sd is {bp_before.std(ddof=1):.1f} mmHg, but the sd of the")
print(f"  CHANGES is only {changes.std(ddof=1):.1f}. The unpaired test uses the large one")
print("  as its noise estimate, so a real 9 mmHg drop can vanish into non-significance.")
print("\nWilcoxon signed-rank (if Normality were doubtful): "
      f"p = {stats.wilcoxon(bp_after, bp_before).pvalue:.6f}")

**Exercise 4 (challenge).** Your A/B test on conversion revenue shows p = 0.08 after one
week, and your manager asks whether to "let it run a bit longer until it's significant".
(a) Explain the statistical problem with that plan.
(b) Simulate it: run a test that checks for significance every day for 30 days on data
where **there is no real effect**, and report how often it eventually "wins".
(c) What is the correct approach?

In [ ]:
# --- Solution 4 -------------------------------------------------------------
# (b) Simulate "peeking": test daily and stop as soon as p < 0.05.
def peeking_experiment(daily_n=100, days=30, effect=0.0, alpha=0.05):
    a = np.empty(0); b = np.empty(0)
    for day in range(days):
        a = np.r_[a, rng.normal(0, 1, daily_n)]
        b = np.r_[b, rng.normal(effect, 1, daily_n)]
        if stats.ttest_ind(a, b, equal_var=False).pvalue < alpha:
            return True, day + 1
    return False, days

sims = 400
wins = [peeking_experiment() for _ in range(sims)]
false_pos_rate = np.mean([w[0] for w in wins])

single_look = np.mean([
    stats.ttest_ind(rng.normal(0, 1, 3000), rng.normal(0, 1, 3000), equal_var=False).pvalue < 0.05
    for _ in range(sims)
])

print("(a) Every time you look, you get another chance to cross the threshold by luck.")
print("    The advertised 5% error rate applies to ONE pre-planned analysis, not 30.\n")
print(f"(b) With NO real effect at all:")
print(f"    peeking daily for 30 days -> 'significant' in {false_pos_rate:.1%} of experiments")
print(f"    a single look at the end  -> 'significant' in {single_look:.1%} of experiments")
print(f"    Peeking inflated the false-positive rate {false_pos_rate/max(single_look,1e-9):.1f}-fold.\n")
print("(c) The correct approaches, in order of preference:")
print("    1. Fix the sample size in advance by a power calculation, then look ONCE.")
print("    2. If you must monitor, use a sequential design with alpha-spending")
print("       (O'Brien-Fleming, Pocock) or a Bayesian/always-valid approach.")
print("    3. Report the confidence interval, not just the verdict -- 'p = 0.08 with a CI")
print("       of -1% to +9%' honestly says 'we do not know yet, and here is how much'.")
print("    Never extend a test BECAUSE it is not yet significant.")

---
## Summary

| Test | Question | scipy call | df |
|---|---|---|---|
| One-sample t | Is the mean equal to $\mu_0$? | `ttest_1samp(x, mu0)` | $n-1$ |
| Student two-sample | Do two group means differ (equal variances)? | `ttest_ind(a, b)` | $n_1+n_2-2$ |
| **Welch two-sample** | Do two group means differ (any variances)? | `ttest_ind(a, b, equal_var=False)` | Welch–Satterthwaite |
| Paired t | Is the mean change non-zero? | `ttest_rel(after, before)` | $n-1$ |
| Mann–Whitney U | Non-Normal, independent | `mannwhitneyu(a, b)` | — |
| Wilcoxon | Non-Normal, paired | `wilcoxon(after, before)` | — |

**Rules to carry away**

1. Default to **Welch's** test for independent groups
2. Use the **paired** test whenever the data is genuinely paired — the power gain is free
3. Always report the **confidence interval** and the **effect size**, not just $p$
4. Check assumptions with a **Q–Q plot**, not just a Normality test
5. Fix your sample size in advance; **never peek and extend**

**Next up:** [Notebook 8 — Chi-Square Tests](8.%20Chi-Square%20Test.ipynb), for categorical
data, where means do not exist and counts are all you have.